Step 1 — Verify PyTorch and RTX 3080

In [ ]:
#cell 1
import sys
import importlib.util

print("Python version:", sys.version.split()[0])

try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
        print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
    else:
        print("GPU acceleration unavailable")

except ImportError:
    print("PyTorch is not installed in this environment")

for package in ["timm", "torchvision", "pandas", "sklearn", "PIL", "matplotlib", "tqdm"]:
    status = "Installed" if importlib.util.find_spec(package) else "Missing"
    print(f"{package}: {status}")


yCharm is currently using a CPU-only build of PyTorch. I have to utilize my gpu so Installing CUDA-enabled PyTorch

In [ ]:
#cell 2
%pip install --upgrade --force-reinstall torch==2.9.0 torchvision==0.24.0 --index-url https://download.pytorch.org/whl/cu128


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Step 2 — Install the remaining libraries

In [ ]:
#cell 3
%pip install timm==1.0.22


Step 3 — Verifying GPU and ViT dependencies

In [ ]:
#cell 4
import torch
import torchvision
import timm

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("timm:", timm.__version__)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=3
).to("cuda")

print("ViT initialized successfully")
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

del model
torch.cuda.empty_cache()


This verifies that we can initialize a three-class Vision Transformer directly on your GPU. We're using pretrained=False only for this initialization test; actual fine-tuning will use pretrained weights. WE GOT 85,800,963 ViT parameters. 

Step - 4 Load and validate the CSV manifests

In [ ]:
#cell 5
from pathlib import Path
import pandas as pd

ROOT = Path(r"PROJECT_ROOT")
METADATA_DIR = ROOT / "NetraScan_Metadata"

train_df = pd.read_csv(METADATA_DIR / "netrascan_train.csv")
val_df = pd.read_csv(METADATA_DIR / "netrascan_val.csv")
test_df = pd.read_csv(METADATA_DIR / "netrascan_test.csv")

print("TRAIN:", train_df.shape)
print("VALIDATION:", val_df.shape)
print("TEST:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nClass distribution:")
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{name}")
    print(df["class"].value_counts())

print("\nChecking image paths:")
for name, df in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    missing = sum(not Path(p).is_file() for p in df["image_path"])
    print(f"{name}: {missing} missing images")

print("\nChecking visual-family overlap:")
train_families = set(train_df["visual_family_id"])
val_families = set(val_df["visual_family_id"])
test_families = set(test_df["visual_family_id"])

print("Train/Val:", len(train_families & val_families))
print("Train/Test:", len(train_families & test_families))
print("Val/Test:", len(val_families & test_families))


All dataset validation checks passed. We're ready to construct the actual PyTorch image pipeline for ViT-B/16.

Step 5 — Building the PyTorch Dataset and DataLoaders

We'll use timm to obtain the pretrained ViT's recommended image normalization, with mild augmentation applied only to training images.

Dataset class and preprocessing

In [ ]:
#Dataset class and preprocessing
#cell 6


import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from timm.data import resolve_model_data_config

CLASS_NAMES = ["normal", "immature", "mature"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

MODEL_NAME = "vit_base_patch16_224"

reference_model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=3
)

data_config = resolve_model_data_config(reference_model)
del reference_model

image_size = data_config["input_size"][-1]
mean = data_config["mean"]
std = data_config["std"]

train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomRotation(8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

eval_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

class CataractDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(row["image_path"]).convert("RGB")
        image = self.transform(image)
        label = CLASS_TO_IDX[row["class"]]
        return image, label

train_dataset = CataractDataset(train_df, train_transform)
val_dataset = CataractDataset(val_df, eval_transform)
test_dataset = CataractDataset(test_df, eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Classes:", CLASS_TO_IDX)
print("Image size:", image_size)
print("Normalization mean:", mean)
print("Normalization std:", std)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


HERE THIS,
Creates a reusable CataractDataset class that loads images directly from the existing CSV paths.
Maps the three classes to numerical labels: Normal = 0, Immature = 1, Mature = 2.
Resizes images to the ViT input resolution, normally 224 × 224.
Applies mild geometric and color augmentation to training images only.
Uses deterministic preprocessing for validation and testing.
Creates DataLoaders with a batch size of 16, suitable as an initial setting for your RTX 3080.

Step 6 — Visualize a training batch

In [ ]:
#cell 7 Display sample images

import matplotlib.pyplot as plt
import numpy as np

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image dtype:", images.dtype)
print("Label indices:", labels.tolist())

fig, axes = plt.subplots(2, 4, figsize=(13, 7))

mean_array = np.array(mean).reshape(3, 1, 1)
std_array = np.array(std).reshape(3, 1, 1)

for i, ax in enumerate(axes.flat):
    image = images[i].permute(1, 2, 0).numpy()
    image = image * std_array.reshape(1, 1, 3) + mean_array.reshape(1, 1, 3)
    image = np.clip(image, 0, 1)

    ax.imshow(image)
    ax.set_title(IDX_TO_CLASS[labels[i].item()])
    ax.axis("off")

plt.suptitle("NetraScan.AI — Training Batch Inspection", fontsize=15)
plt.tight_layout()
plt.show()


The images have noticeably different lighting, sharpness, and pupil appearances. This is expected with multiple source datasets, but it reinforces why we need robust validation.Also, the class labels in the visualization are very close to the image boundaries. That's only a plotting-layout issue, not a dataset problem.

Step 7- Initialize our pretrained ViT-B/16 ready to load ImageNet-pretrained weights and replace the classification head with three output classes.

In [ ]:
#CELL 8
import torch
import timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

MODEL_NAME = "vit_base_patch16_224"

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=3
)

model = model.to(DEVICE)

print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Output classes:", model.num_classes)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Verify the forward pass using a small batch
model.eval()

with torch.inference_mode():
    sample_batch = images[:2].to(DEVICE)
    outputs = model(sample_batch)

print("Input shape:", sample_batch.shape)
print("Output shape:", outputs.shape)

assert outputs.shape == (2, 3)

print("Pretrained ViT-B/16 loaded successfully!")


pretrained ViT-B/16 is loaded on the RTX 3080 and its forward pass works correctly.

The Hugging Face symlink warning is harmless. It only means Windows may use additional disk space for cached model files. No action is needed.

STEP -8 Configure ViT fine-tuning
We'll use AdamW, cross-entropy loss, mixed precision, and validation-based checkpoint selection.

Our initial settings:
Hyperparameter	Value
Architecture	ViT-B/16
Pretraining	ImageNet
Optimizer	AdamW
Learning rate	2 × 10⁻⁵
Weight decay	0.01
Batch size	16
Maximum epochs	10
Early stopping patience	3
Mixed precision	FP16
Model selection	Validation macro-F1

In [ ]:
#CELL 9 Training configuration

import random
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
from torch.amp import GradScaler

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 3

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler("cuda")

CHECKPOINT_DIR = ROOT / "models" / "vit"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = ROOT / "results" / "vit"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Training configuration ready")
print("Device:", DEVICE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", train_loader.batch_size)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)


Our model has approx 85.8 million parameters trainable. Full fine tuning with Adam will be very costly computationally.
so we will first benchmark a single training step. If batch size 16 causes CUDA out-of-memory, we'll reduce it to 8 and use gradient accumulation if needed.

Before starting a full epoch, also need to confirm that whether my RTX 3080 12 GB can handle the backward pass with batch size 16. 

Step 9 — GPU memory benchmark

This test performs one actual training step using mixed precision (FP16), including forward propagation, loss calculation, backpropagation, and an optimizer update. This benchmark performs one optimizer update. That's fine, but it means the model is no longer at its exact pretrained initialization. Before the full experiment, we'll recreate the model and optimizer so training starts from a clean, reproducible state.

In [ ]:
#cell 10 single step training gpu benchmark test 

import time
from torch.amp import autocast

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.train()

batch_images, batch_labels = next(iter(train_loader))
batch_images = batch_images.to(DEVICE, non_blocking=True)
batch_labels = batch_labels.to(DEVICE, non_blocking=True)

print("Batch shape:", batch_images.shape)
print("Starting GPU memory benchmark...")

try:
    torch.cuda.synchronize()
    start_time = time.perf_counter()

    optimizer.zero_grad(set_to_none=True)

    with autocast(device_type="cuda", dtype=torch.float16):
        outputs = model(batch_images)
        loss = criterion(outputs, batch_labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)
    reserved_memory = torch.cuda.max_memory_reserved() / (1024 ** 3)

    print("\nBENCHMARK SUCCESSFUL")
    print(f"Loss: {loss.item():.4f}")
    print(f"Step time: {elapsed:.2f} seconds")
    print(f"Peak allocated VRAM: {peak_memory:.2f} GB")
    print(f"Peak reserved VRAM: {reserved_memory:.2f} GB")
    print(f"GPU total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB")

except torch.cuda.OutOfMemoryError:
    print("\nCUDA OUT OF MEMORY")
    print("Batch size 16 is too large for this configuration.")
    print("We will reduce the batch size to 8.")

finally:
    optimizer.zero_grad(set_to_none=True)

    del batch_images, batch_labels

    if "outputs" in globals():
        del outputs

    torch.cuda.empty_cache()



Our first ViT-B/16 training step completed successfully on the RTX 3080. Forward propagation, backpropagation, and the AdamW
optimizer update all worked. 

Training step -0.29s
Peak allocated VRAM- 1.55 GB
Initial loss- 1.4977

The reported VRAM is unusually low for full ViT-B/16 fine-tuning with AdamW, so we'll treat it as a successful smoke test rather than a definitive memory benchmark. The first-step timing also isn't sufficient to predict full-epoch duration.

Step 10 — Implement training and validation functions. Also We'll now prepare reusable functions for training, validation, accuracy, macro-F1, and GPU memory monitoring.

Important: The benchmark already modified the model weights. Before the real training run, we'll reload the original pretrained weights and recreate the optimizer and GradScaler.

In [ ]:
#cell 11 Training and validation functions

import time
import torch
import numpy as np

from torch.amp import autocast
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm


def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    progress = tqdm(loader, desc="Training", leave=False)

    for batch_images, batch_labels in progress:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * batch_images.size(0)

        predictions = outputs.detach().argmax(dim=1)

        all_preds.extend(predictions.cpu().tolist())
        all_labels.extend(batch_labels.cpu().tolist())

        progress.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels, all_preds, average="macro", zero_division=0
    )

    return epoch_loss, epoch_acc, epoch_f1


@torch.inference_mode()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    progress = tqdm(loader, desc="Validation", leave=False)

    for batch_images, batch_labels in progress:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)

        total_loss += loss.item() * batch_images.size(0)

        predictions = outputs.argmax(dim=1)

        all_preds.extend(predictions.cpu().tolist())
        all_labels.extend(batch_labels.cpu().tolist())

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels, all_preds, average="macro", zero_division=0
    )

    return epoch_loss, epoch_acc, epoch_f1


print("Training and validation functions initialized.")


What this does
- Trains using mixed-precision FP16.
- Calculates sample-weighted epoch loss.
- Measures accuracy and macro-F1.
- Uses model.eval() and disables gradients during validation.
- Keeps validation data separate from training.
- Displays batch-level progress using tqdm.
Macro-F1 is particularly relevant because we need balanced performance across Normal, Immature, and Mature cataract classes.

Now the ViT-B/16 training and validation functions are ready. Now we'll launch our first proper training experiment.

Before starting, we must reset the model because our GPU benchmark already performed one optimizer update.

Step 11 — Initialize the experiment runner:
- Reload the original pretrained ViT-B/16 weights.
- Reset the AdamW optimizer and mixed-precision scaler.
- Train for up to 10 epochs.
- Calculate training and validation accuracy, loss, and macro-F1.
- Save the best checkpoint based on validation macro-F1.
- Apply early stopping after 3 epochs without improvement.
- Save experiment history and configuration for our research paper.

In [ ]:
#cell 12 Full ViT training

import json
import random
import time
import pandas as pd
import numpy as np
import torch
import timm

from torch.optim import AdamW
from torch.amp import GradScaler

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reset the model after our benchmark
del model
torch.cuda.empty_cache()

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=3
).to(DEVICE)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = GradScaler("cuda")

# Save experiment configuration
config = {
    "model": MODEL_NAME,
    "pretrained": True,
    "classes": CLASS_NAMES,
    "class_to_idx": CLASS_TO_IDX,
    "image_size": image_size,
    "normalization_mean": list(mean),
    "normalization_std": list(std),
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
    "batch_size": train_loader.batch_size,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": EPOCHS,
    "early_stopping_patience": PATIENCE,
    "seed": SEED,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "mixed_precision": "FP16",
    "selection_metric": "validation_macro_f1"
}

with open(RESULTS_DIR / "experiment_config.json", "w") as f:
    json.dump(config, f, indent=4)

best_f1 = -1.0
patience_counter = 0
history = []

print("Starting ViT-B/16 fine-tuning")
print("Device:", DEVICE)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("-" * 65)

for epoch in range(EPOCHS):
    epoch_start = time.perf_counter()

    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, DEVICE
    )

    val_loss, val_acc, val_f1 = validate_one_epoch(
        model, val_loader, criterion, DEVICE
    )

    elapsed = time.perf_counter() - epoch_start

    metrics = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
        "epoch_seconds": elapsed
    }

    history.append(metrics)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Train F1: {train_f1:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    # Save history after every epoch
    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "training_history.csv",
        index=False
    )

    if val_f1 > best_f1 + 1e-4:
        best_f1 = val_f1
        patience_counter = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_macro_f1": val_f1,
                "class_to_idx": CLASS_TO_IDX,
                "model_name": MODEL_NAME
            },
            CHECKPOINT_DIR / "vit_best.pth"
        )

        print(f"  Best checkpoint saved — Val Macro-F1: {best_f1:.4f}")

    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print("-" * 65)
print("Training completed.")
print(f"Best validation macro-F1: {best_f1:.4f}")
print("Checkpoint:", CHECKPOINT_DIR / "vit_best.pth")
print("History:", RESULTS_DIR / "training_history.csv")


We need to investigate the 100% accuracy
Getting perfect validation accuracy after just three epochs is unusual for a medical image classification task.

It could indicate that our dataset is relatively easy, but it could also indicate:
- Augmented versions of the same original images appearing across splits.
- Near-duplicate images that our pHash=0 grouping didn't capture.
- Source-specific visual patterns that make classification artificially easy.

In [ ]:
# cell 13 Audit cross-split near-duplicates

from collections import Counter
import numpy as np
import pandas as pd

all_splits = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

hashes = np.array(
    [int(h, 16) for h in all_splits["phash"]],
    dtype=np.uint64
)

split_labels = all_splits["split"].to_numpy()

cross_split_pairs = []

for i in range(len(hashes)):
    for j in range(i + 1, len(hashes)):
        if split_labels[i] == split_labels[j]:
            continue

        distance = (
            int(hashes[i]) ^ int(hashes[j])
        ).bit_count()

        if distance <= 4:
            cross_split_pairs.append({
                "index_1": i,
                "index_2": j,
                "split_1": split_labels[i],
                "split_2": split_labels[j],
                "class_1": all_splits.iloc[i]["class"],
                "class_2": all_splits.iloc[j]["class"],
                "distance": distance
            })

audit_df = pd.DataFrame(cross_split_pairs)

print("Cross-split near-duplicate candidates:", len(audit_df))

if not audit_df.empty:
    print("\nDistance distribution:")
    print(audit_df["distance"].value_counts().sort_index())

    print("\nClass agreement:")
    print(
        (audit_df["class_1"] == audit_df["class_2"])
        .value_counts()
    )

    display(audit_df.head(10))


This will identify images with perceptual-hash distances of 0, 2, or 4 across different splits.

A low pHash distance is only a candidate similarity signal, not proof that two images are duplicates. We'll inspect suspicious pairs visually before changing anything.

Our audit identified 219 cross-split image pairs that may be visually related.
Audit results:
Metric	Result:
Cross-split candidate pairs	= 219
pHash distance = 0	-0
pHash distance = 2	-102
pHash distance = 4	-117
Same-class pairs	-219
Different-class pairs	-0 

The fact that all 219 pairs have matching labels is particularly interesting.

However, 219 pairs does not mean 219 leaked images. Multiple pairs may involve the same image, and perceptual similarity alone doesn't establish that two photographs share the same original.

Our earlier visual-family grouping successfully prevented pHash=0 overlap, but it didn't eliminate these weaker similarity candidates.

Before we change the dataset or retrain ViT, we need to see what these pairs actually look like.

Visually inspect suspicious image pairs 

We need to distinguish two cases:

Case A — Same original photograph

If the two images have identical eye anatomy, reflections, eyelashes, and background features, but differ in brightness, rotation, compression, or cropping, they are likely augmented versions of the same photograph.

That would represent a genuine split-integrity problem requiring correction.

Case B — Different photographs with similar appearances

Two images may have similar pupil shapes, lighting, and cataract opacity while representing different eyes.

Those should not automatically be merged based on pHash distance alone.
Our next decision
If we confirm actual cross-split duplicates, we'll rebuild the visual-family groups using verified relationships and regenerate the manifests. Then we'll retrain ViT, Swin, and MobileViT using the corrected splits.

In [ ]:
#cell 14 Cross-split visual inspection

import matplotlib.pyplot as plt
from PIL import Image

# Prioritize the closest perceptual-hash matches
inspection_df = (
    audit_df
    .sort_values("distance")
    .drop_duplicates(subset=["index_1"])
    .head(12)
)

fig, axes = plt.subplots(
    len(inspection_df), 2,
    figsize=(9, len(inspection_df) * 3)
)

for row_num, (_, pair) in enumerate(inspection_df.iterrows()):
    idx1 = int(pair["index_1"])
    idx2 = int(pair["index_2"])

    record1 = all_splits.iloc[idx1]
    record2 = all_splits.iloc[idx2]

    for col, record in enumerate([record1, record2]):
        image = Image.open(record["image_path"]).convert("RGB")

        axes[row_num, col].imshow(image)
        axes[row_num, col].axis("off")

        axes[row_num, col].set_title(
            f"{record['split']} | {record['class']}\n"
            f"{record['source']} | {record['filename'][:30]}",
            fontsize=8
        )

    axes[row_num, 0].set_ylabel(
        f"pHash distance: {int(pair['distance'])}",
        fontsize=9
    )

plt.suptitle(
    "NetraScan.AI — Cross-Split Near-Duplicate Inspection",
    fontsize=14
)

plt.tight_layout()
plt.show()


Several pairs are unmistakably the same original eye photograph, appearing in different partitions with slightly different compression, brightness, or augmentation.

For example, I can see:
- The same eye with a distinctive window reflection appearing in training and testing.
- The same normal eye appearing in both training and validation.
- Identical pupil shapes, reflections, eyelids, and surrounding anatomical features across several pairs.
- Matching original filename stems such as 11_JPG, 4_JPG, and 8_JPG.
This explains why our ViT achieved 100% validation accuracy so quickly, although leakage may not be the only contributing factor.
What this means for our research
Our original pHash=0 grouping was too conservative. It successfully prevented identical perceptual hashes from crossing splits, but it missed augmented variants with pHash distances of 2–4.

The 219 suspicious pairs are not necessarily 219 distinct leaked images. Nevertheless, the visually confirmed examples are sufficient to establish that our existing split is compromised.

We must correct the dataset before training Swin or MobileViT.

The existing ViT checkpoint remains a preliminary experiment, but its validation score should not be used as our final research result.

Identify augmentation families using filenames Before rebuilding the split, I want to inspect the original filenames systematically.

For Roboflow images, filenames often follow patterns such as:

11_JPG.rf.<augmentation_hash>.jpg

Multiple files sharing the same original stem may be derived from the same photograph, even when their pHashes differ.

We'll first analyze these filename families without changing anything.

In [ ]:
#cell 15 Filename lineage audit

import re
import pandas as pd

lineage_df = all_splits.copy()

def extract_original_stem(row):
    filename = str(row["filename"])
    source = str(row["source"])

    stem = filename.rsplit(".", 1)[0]

    if source == "cataract_v01":
        stem = re.sub(
            r"\.rf\.[a-fA-F0-9]+$",
            "",
            stem
        )

    return stem.lower()

lineage_df["original_stem"] = lineage_df.apply(
    extract_original_stem,
    axis=1
)

# Include class and source to avoid unrelated filename collisions
lineage_df["lineage_key"] = (
    lineage_df["source"].astype(str)
    + "::"
    + lineage_df["class"].astype(str)
    + "::"
    + lineage_df["original_stem"]
)

family_summary = (
    lineage_df.groupby("lineage_key")
    .agg(
        images=("filename", "size"),
        splits=("split", "nunique"),
        sources=("source", "nunique")
    )
    .reset_index()
)

cross_split_lineages = family_summary[
    family_summary["splits"] > 1
]

print("Total filename lineage groups:", len(family_summary))
print("Cross-split filename lineage groups:", len(cross_split_lineages))

print("\nImages in cross-split lineages:")
print(
    lineage_df[
        lineage_df["lineage_key"].isin(
            cross_split_lineages["lineage_key"]
        )
    ].shape[0]
)

display(
    cross_split_lineages
    .sort_values("images", ascending=False)
    .head(20)
)


We've now confirmed a second source of dataset leakage: augmentation lineage. This is an important finding for the reliability of our NetraScan.AI experiments.
Step 16 — Results
Metric	Result
Total filename lineage groups	3,085
Cross-split lineage groups	42
Images belonging to those groups	218
Largest groups shown	8 images
Cross-split pHash candidate pairs (previous audit)	219

Your screenshot shows examples such as cataract_v01::immature::20_jpg and cataract_v01::mature::30_jpg, whose variants are distributed across multiple splits.

This means our original splitting strategy separated some augmented versions of the same original photograph.

We need to fix this before treating our ViT results as research-grade evidence.
What we're going to do
Rather than delete hundreds of images or manually change their labels, we'll reconstruct visual families using three sources of evidence:
1. Existing pHash=0 visual-family relationships.
2. Original filename lineage within Cataract v01.
3. High-confidence cross-source image relationships.
Then we'll regenerate the train/validation/test manifests so related images remain together.

We'll preserve the existing CSV files and the preliminary ViT checkpoint.

Step 17 — Building stronger image-family groups

In [ ]:
#cell 16 Union-Find grouping with filename lineage

from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Start from our existing combined dataset
corrected_df = lineage_df.copy().reset_index(drop=True)

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return

        if self.rank[root_a] < self.rank[root_b]:
            root_a, root_b = root_b, root_a

        self.parent[root_b] = root_a

        if self.rank[root_a] == self.rank[root_b]:
            self.rank[root_a] += 1


uf = UnionFind(len(corrected_df))

# 1. Preserve original pHash=0 visual families
for _, group in corrected_df.groupby("visual_family_id"):
    indices = group.index.tolist()
    for idx in indices[1:]:
        uf.union(indices[0], idx)

# 2. Merge Roboflow augmentation variants sharing original filename
roboflow_groups = corrected_df[
    corrected_df["source"] == "cataract_v01"
].groupby("lineage_key")

for _, group in roboflow_groups:
    indices = group.index.tolist()
    for idx in indices[1:]:
        uf.union(indices[0], idx)

# 3. Merge pHash distance <= 2 within matching classes
hashes = [
    int(h, 16) for h in corrected_df["phash"]
]
labels = corrected_df["class"].tolist()

merged_pairs = 0

for i in range(len(corrected_df)):
    for j in range(i + 1, len(corrected_df)):
        if labels[i] != labels[j]:
            continue

        distance = (hashes[i] ^ hashes[j]).bit_count()

        if distance <= 2:
            uf.union(i, j)
            merged_pairs += 1

# Assign new visual-family IDs
roots = [uf.find(i) for i in range(len(corrected_df))]
unique_roots = sorted(set(roots))

root_to_id = {
    root: f"VF2_{idx + 1:05d}"
    for idx, root in enumerate(unique_roots)
}

corrected_df["visual_family_id_v2"] = [
    root_to_id[root] for root in roots
]

family_sizes = corrected_df[
    "visual_family_id_v2"
].value_counts()

corrected_df["visual_family_size_v2"] = (
    corrected_df["visual_family_id_v2"].map(family_sizes)
)

print("Total images:", len(corrected_df))
print("Original visual families:", corrected_df["visual_family_id"].nunique())
print("Corrected visual families:", corrected_df["visual_family_id_v2"].nunique())
print("pHash <= 2 candidate pairs:", merged_pairs)
print("Largest corrected family:", family_sizes.max())

conflicts = (
    corrected_df.groupby("visual_family_id_v2")["class"]
    .nunique()
)

print("Families with conflicting labels:", (conflicts > 1).sum())

# Check how many corrected families span original splits
cross_split_families = (
    corrected_df.groupby("visual_family_id_v2")["split"]
    .nunique()
)

print(
    "Corrected families spanning original splits:",
    (cross_split_families > 1).sum()
)


We're using pHash distance ≤2 as a provisional high-similarity grouping criterion, but perceptual hashes can still produce false matches. This is a practical correction, not proof of patient-level independence.

We are deliberately not merging every distance-4 pair, because our earlier visual inspection showed that some visually similar eyes are genuinely different photographs.

After this grouping, we'll verify class consistency and examine the remaining cross-split candidates before finalizing any new manifests.

Our corrected visual-family grouping has successfully identified additional relationships that the original pipeline missed.
Step 17 — Results
Metric	                                       Original	          CorrectedV2
Total images	                                   3,542	      3,542
Visual families	                                   3,127	      2,957
Largest family	                                   9	          10
Families with conflicting labels	               0	          0
Families spanning old splits	                   —	          63

The key finding is that 63 corrected visual families span multiple partitions in our original train/validation/test split.

That confirms why we need to regenerate the manifests.

However, these are still provisional groups based on filename lineage and perceptual similarity. We should validate the grouping assumptions before treating the new split as final.

Step 18 — Generate corrected train/validation/test splits
We'll now create a fresh stratified split using the corrected visual families.

Our target remains:
- Training: 70%
- Validation: 15%
- Testing: 15%
Every image belonging to the same corrected visual family must remain in the same partition.

In [ ]:
#cell 17 Corrected group-aware splitting


from sklearn.model_selection import train_test_split
import pandas as pd

# One representative record per corrected visual family
family_df = (
    corrected_df.groupby("visual_family_id_v2")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v2", "first")
    )
    .reset_index()
)

# 70% train, 30% temporary
train_families, temp_families = train_test_split(
    family_df,
    test_size=0.30,
    stratify=family_df["class_label"],
    random_state=42
)

# Divide remaining 30% into 15% validation and 15% test
val_families, test_families = train_test_split(
    temp_families,
    test_size=0.50,
    stratify=temp_families["class_label"],
    random_state=42
)

# Map family IDs to their assigned splits
split_mapping = {}

for family_id in train_families["visual_family_id_v2"]:
    split_mapping[family_id] = "train"

for family_id in val_families["visual_family_id_v2"]:
    split_mapping[family_id] = "val"

for family_id in test_families["visual_family_id_v2"]:
    split_mapping[family_id] = "test"

corrected_df["split_v2"] = (
    corrected_df["visual_family_id_v2"].map(split_mapping)
)

print("Corrected split distribution:")
print(corrected_df["split_v2"].value_counts())

print("\nClass distribution:")
print(
    pd.crosstab(
        corrected_df["split_v2"],
        corrected_df["class"]
    )
)

# Verify corrected visual families never cross splits
leakage_check = (
    corrected_df.groupby("visual_family_id_v2")["split_v2"]
    .nunique()
)

print("\nCorrected families crossing splits:",
      (leakage_check > 1).sum())

assert (leakage_check > 1).sum() == 0

print("\nPASS — No corrected visual-family overlap across splits.")


ur V2 dataset split has passed the visual-family isolation check. We've successfully reorganized all 3,542 images without deleting any.

Step 19 — Audit remaining cross-split similarities

In [ ]:
#cell 18 V2 split-integrity audit


from collections import Counter
import pandas as pd

# Work only with the corrected in-memory dataset
v2_audit = corrected_df.reset_index(drop=True).copy()

hashes_v2 = [
    int(h, 16) for h in v2_audit["phash"]
]

remaining_pairs = []

for i in range(len(v2_audit)):
    for j in range(i + 1, len(v2_audit)):
        if v2_audit.at[i, "split_v2"] == v2_audit.at[j, "split_v2"]:
            continue

        distance = (
            hashes_v2[i] ^ hashes_v2[j]
        ).bit_count()

        if distance <= 4:
            remaining_pairs.append({
                "index_1": i,
                "index_2": j,
                "distance": distance,
                "split_1": v2_audit.at[i, "split_v2"],
                "split_2": v2_audit.at[j, "split_v2"],
                "class_1": v2_audit.at[i, "class"],
                "class_2": v2_audit.at[j, "class"],
                "source_1": v2_audit.at[i, "source"],
                "source_2": v2_audit.at[j, "source"]
            })

remaining_df = pd.DataFrame(remaining_pairs)

print("Remaining cross-split pHash candidates:", len(remaining_df))

if not remaining_df.empty:
    print("\nDistance distribution:")
    print(remaining_df["distance"].value_counts().sort_index())

    print("\nClass agreement:")
    print(
        (remaining_df["class_1"] == remaining_df["class_2"])
        .value_counts()
    )

    print("\nSource combinations:")
    print(
        pd.crosstab(
            remaining_df["source_1"],
            remaining_df["source_2"]
        )
    )

    display(remaining_df.head(15))

# Check Roboflow augmentation lineage
roboflow_v2 = v2_audit[
    v2_audit["source"] == "cataract_v01"
]

lineage_overlap = (
    roboflow_v2.groupby("lineage_key")["split_v2"]
    .nunique()
)

print(
    "\nRoboflow lineages crossing V2 splits:",
    int((lineage_overlap > 1).sum())
)

print(
    "V2 families crossing splits:",
    int(
        (
            v2_audit.groupby("visual_family_id_v2")["split_v2"]
            .nunique() > 1
        ).sum()
    )
)


ur V2 split has eliminated all cross-split pHash matches at distances 0 and 2, and all detected Roboflow filename-lineage overlaps. However, 120 suspicious pairs remain.

Step 20 — Inspect the remaining 120 candidates

In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

# Select representative pairs from each dataset
inspection = []

for source in ["cataract_v01", "dataset_katarak"]:
    subset = remaining_df[
        (remaining_df["source_1"] == source) &
        (remaining_df["source_2"] == source)
    ].copy()

    subset["family_1"] = subset["index_1"].map(
        v2_audit["visual_family_id_v2"]
    )
    subset["family_2"] = subset["index_2"].map(
        v2_audit["visual_family_id_v2"]
    )

    subset["family_pair"] = subset.apply(
        lambda row: tuple(sorted([
            row["family_1"],
            row["family_2"]
        ])),
        axis=1
    )

    subset = subset.drop_duplicates("family_pair")
    inspection.append(subset.head(8))

inspection_df = pd.concat(
    inspection, ignore_index=True
)

print("Representative family pairs:", len(inspection_df))

if len(inspection_df) > 0:
    fig, axes = plt.subplots(
        len(inspection_df), 2,
        figsize=(10, len(inspection_df) * 2.8),
        squeeze=False
    )

    for row_num, (_, pair) in enumerate(inspection_df.iterrows()):
        idx1 = int(pair["index_1"])
        idx2 = int(pair["index_2"])

        for col, idx in enumerate([idx1, idx2]):
            record = v2_audit.iloc[idx]

            with Image.open(record["image_path"]) as img:
                image = img.convert("RGB")

            axes[row_num, col].imshow(image)
            axes[row_num, col].axis("off")
            axes[row_num, col].set_title(
                f"{record['source']} | {record['split_v2']}\n"
                f"{record['class']} | {record['filename'][:35]}",
                fontsize=8
            )

        axes[row_num, 0].set_ylabel(
            f"Pair {row_num + 1}\npHash = {pair['distance']}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()


Step 21 — Strengthen the dataset grouping
Since we're preparing a research project, the approach should be like:
1. Merge the confirmed duplicate relationships.
2. Rebuild the family-aware split.
3. Audit the new split again.
4. Only then retrain ViT, Swin, and MobileViT.

In [ ]:

# Build V3 groups from the existing V2 groups
uf_v3 = UnionFind(len(v2_audit))

# Preserve every V2 visual family
for _, group in v2_audit.groupby("visual_family_id_v2"):
    indices = group.index.tolist()

    for idx in indices[1:]:
        uf_v3.union(indices[0], idx)

# Merge all same-class pHash distance-4 candidates
merged_candidates = 0

for _, pair in remaining_df.iterrows():
    i = int(pair["index_1"])
    j = int(pair["index_2"])

    if (
        pair["distance"] == 4
        and pair["class_1"] == pair["class_2"]
    ):
        uf_v3.union(i, j)
        merged_candidates += 1

roots_v3 = [
    uf_v3.find(i)
    for i in range(len(v2_audit))
]

unique_roots_v3 = sorted(set(roots_v3))

root_to_v3 = {
    root: f"VF3_{n + 1:05d}"
    for n, root in enumerate(unique_roots_v3)
}

v3_df = v2_audit.copy()

v3_df["visual_family_id_v3"] = [
    root_to_v3[root]
    for root in roots_v3
]

v3_sizes = v3_df["visual_family_id_v3"].value_counts()

v3_df["visual_family_size_v3"] = (
    v3_df["visual_family_id_v3"].map(v3_sizes)
)

print("Total images:", len(v3_df))
print("V2 families:", v2_audit["visual_family_id_v2"].nunique())
print("V3 families:", v3_df["visual_family_id_v3"].nunique())
print("Candidate pairs grouped:", merged_candidates)
print("Largest V3 family:", v3_sizes.max())

conflicts = (
    v3_df.groupby("visual_family_id_v3")["class"]
    .nunique()
)

print("Conflicting-label families:", (conflicts > 1).sum())

assert (conflicts > 1).sum() == 0

print("\nPASS — V3 grouping constructed successfully.")


V3 grouping is complete, with zero conflicting-label families. We're now much closer to having a defensible dataset for NetraScan.AI.

Step 22 — Generate the V3 train/validation/test split

In [ ]:
#Cell 20 — V3 stratified splitting
from sklearn.model_selection import train_test_split
import pandas as pd

RANDOM_SEED = 42

# One record per V3 visual family
v3_family_df = (
    v3_df.groupby("visual_family_id_v3")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v3", "first")
    )
    .reset_index()
)

# Split families: 70% train, 30% temporary
v3_train_families, v3_temp_families = train_test_split(
    v3_family_df,
    test_size=0.30,
    stratify=v3_family_df["class_label"],
    random_state=RANDOM_SEED
)

# Split temporary families equally into validation and test
v3_val_families, v3_test_families = train_test_split(
    v3_temp_families,
    test_size=0.50,
    stratify=v3_temp_families["class_label"],
    random_state=RANDOM_SEED
)

# Build family-to-split mapping
v3_split_map = {}

for family_id in v3_train_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "train"

for family_id in v3_val_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "val"

for family_id in v3_test_families["visual_family_id_v3"]:
    v3_split_map[family_id] = "test"

# Assign splits to all images
v3_df["split_v3"] = (
    v3_df["visual_family_id_v3"].map(v3_split_map)
)

# Display distributions
print("V3 dataset distribution:")
print(v3_df["split_v3"].value_counts())

print("\nV3 class distribution:")
print(
    pd.crosstab(
        v3_df["split_v3"],
        v3_df["class"]
    )
)

# Verify group isolation
v3_overlap = (
    v3_df.groupby("visual_family_id_v3")["split_v3"]
    .nunique()
)

print(
    "\nV3 families crossing splits:",
    int((v3_overlap > 1).sum())
)

assert (v3_overlap > 1).sum() == 0

print("\nPASS — V3 visual-family split isolation verified.")


Step 23 — Comprehensive V3 leakage audit
This checks all 3,542 images, not just pairs from previous splits.

It will identify:
1. Any same-class pHash distance ≤4 pairs crossing V3 splits.
2. Any Roboflow filename lineages crossing splits.
3. Any V3 families crossing splits.
4. Whether all image paths remain valid.

In [ ]:
#Cell 21 — Final V3 integrity audit

from pathlib import Path
from collections import Counter
import pandas as pd

audit = v3_df.reset_index(drop=True).copy()

hash_values = [int(h, 16) for h in audit["phash"]]
labels = audit["class"].tolist()
splits = audit["split_v3"].tolist()

cross_split_pairs = []
all_candidate_pairs = []

for i in range(len(audit)):
    for j in range(i + 1, len(audit)):

        if labels[i] != labels[j]:
            continue

        distance = (hash_values[i] ^ hash_values[j]).bit_count()

        if distance <= 4:
            pair = {
                "index_1": i,
                "index_2": j,
                "distance": distance,
                "split_1": splits[i],
                "split_2": splits[j]
            }

            all_candidate_pairs.append(pair)

            if splits[i] != splits[j]:
                cross_split_pairs.append(pair)

all_candidates_df = pd.DataFrame(all_candidate_pairs)
cross_split_v3_df = pd.DataFrame(cross_split_pairs)

print("Total same-class pHash <=4 pairs:", len(all_candidates_df))
print("Cross-split pHash <=4 pairs:", len(cross_split_v3_df))

if not cross_split_v3_df.empty:
    print("\nCross-split distance distribution:")
    print(
        cross_split_v3_df["distance"]
        .value_counts()
        .sort_index()
    )

# Roboflow filename lineage
roboflow = audit[audit["source"] == "cataract_v01"]

lineage_overlap = (
    roboflow.groupby("lineage_key")["split_v3"].nunique()
)

print(
    "\nRoboflow lineages crossing splits:",
    int((lineage_overlap > 1).sum())
)

# V3 family overlap
family_overlap = (
    audit.groupby("visual_family_id_v3")["split_v3"].nunique()
)

print(
    "V3 families crossing splits:",
    int((family_overlap > 1).sum())
)

# Missing images
missing_paths = sum(
    not Path(path).is_file()
    for path in audit["image_path"]
)

print("Missing image paths:", missing_paths)

print("\nAudit complete.")


The 55 remaining pairs are similarity candidates, not necessarily 55 genuine duplicates. However, since we've already visually confirmed augmentation-related leakage, we should prevent these relationships from crossing partitions.

Step 24 — Build global visual-family groups
Instead of grouping only cross-split pairs, we'll build connected components using every same-class pHash ≤4 relationship across the entire dataset, together with our existing V3 families.

This ensures that when we generate a new split, none of these candidate relationships can cross partitions.

We'll call this our V4 candidate-family grouping.

In [ ]:
#Cell 22 — Global Union-Find grouping

# GLOBAL VISUAL-FAMILY GROUPING
# No files will be modified or overwritten.

global_df = v3_df.reset_index(drop=True).copy()

uf_global = UnionFind(len(global_df))

# Preserve existing V3 families
for _, group in global_df.groupby("visual_family_id_v3"):
    indices = group.index.tolist()

    for idx in indices[1:]:
        uf_global.union(indices[0], idx)

# Merge all same-class pHash <=4 candidate relationships
global_pairs = 0

for _, pair in all_candidates_df.iterrows():
    i = int(pair["index_1"])
    j = int(pair["index_2"])

    # Verify indices and labels
    assert global_df.at[i, "class"] == global_df.at[j, "class"]

    uf_global.union(i, j)
    global_pairs += 1

# Assign global family IDs
roots = [uf_global.find(i) for i in range(len(global_df))]

root_to_global_id = {
    root: f"VFG_{n + 1:05d}"
    for n, root in enumerate(sorted(set(roots)))
}

global_df["visual_family_id_v4"] = [
    root_to_global_id[root]
    for root in roots
]

family_sizes = global_df["visual_family_id_v4"].value_counts()

global_df["visual_family_size_v4"] = (
    global_df["visual_family_id_v4"].map(family_sizes)
)

# Validate class consistency
class_conflicts = (
    global_df.groupby("visual_family_id_v4")["class"]
    .nunique()
)

print("Total images:", len(global_df))
print("V3 families:", global_df["visual_family_id_v3"].nunique())
print("V4 families:", global_df["visual_family_id_v4"].nunique())
print("Candidate pairs processed:", global_pairs)
print("Largest V4 family:", family_sizes.max())
print("Families with conflicting labels:", int((class_conflicts > 1).sum()))

print("\nV4 family size distribution:")
print(family_sizes.value_counts().sort_index().tail(15))

assert (class_conflicts > 1).sum() == 0

print("\nPASS — Global candidate-family grouping complete.")


Step 25 — Generate the V4 train/validation/test split
Now we'll split the dataset using these global families.

The target remains 70% training, 15% validation, and 15% testing.

In [ ]:
#Cell 23 — V4 stratified splitting and verification

from sklearn.model_selection import train_test_split
from pathlib import Path
import pandas as pd

RANDOM_SEED = 42

# One representative row per V4 family
v4_family_df = (
    global_df.groupby("visual_family_id_v4")
    .agg(
        class_label=("class", "first"),
        family_size=("visual_family_size_v4", "first")
    )
    .reset_index()
)

# 70% train / 30% temporary
train_groups, temp_groups = train_test_split(
    v4_family_df,
    test_size=0.30,
    stratify=v4_family_df["class_label"],
    random_state=RANDOM_SEED
)

# 15% validation / 15% test
val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    stratify=temp_groups["class_label"],
    random_state=RANDOM_SEED
)

split_map = {}

for group_id in train_groups["visual_family_id_v4"]:
    split_map[group_id] = "train"

for group_id in val_groups["visual_family_id_v4"]:
    split_map[group_id] = "val"

for group_id in test_groups["visual_family_id_v4"]:
    split_map[group_id] = "test"

v4_df = global_df.copy()

v4_df["split_v4"] = (
    v4_df["visual_family_id_v4"].map(split_map)
)

print("V4 split distribution:")
print(v4_df["split_v4"].value_counts())

print("\nV4 class distribution:")
print(pd.crosstab(v4_df["split_v4"], v4_df["class"]))

# 1. Visual-family overlap
family_overlap = (
    v4_df.groupby("visual_family_id_v4")["split_v4"]
    .nunique()
)

# 2. Cross-split pHash <=4 candidates
# all_candidates_df indices correspond to global_df/v4_df
cross_split_candidates = sum(
    v4_df.iloc[int(row["index_1"])]["split_v4"]
    != v4_df.iloc[int(row["index_2"])]["split_v4"]
    for _, row in all_candidates_df.iterrows()
)

# 3. Roboflow augmentation lineage
roboflow_v4 = v4_df[
    v4_df["source"] == "cataract_v01"
]

lineage_overlap = (
    roboflow_v4.groupby("lineage_key")["split_v4"]
    .nunique()
)

# 4. Missing image paths
missing_images = sum(
    not Path(path).is_file()
    for path in v4_df["image_path"]
)

print("\nINTEGRITY CHECKS")
print("-" * 45)

print("Families crossing splits:",
      int((family_overlap > 1).sum()))

print("pHash <=4 pairs crossing splits:",
      cross_split_candidates)

print("Roboflow lineages crossing splits:",
      int((lineage_overlap > 1).sum()))

print("Missing image paths:", missing_images)

assert (family_overlap > 1).sum() == 0
assert cross_split_candidates == 0
assert (lineage_overlap > 1).sum() == 0
assert missing_images == 0

print("\nPASS — V4 split integrity checks successful.")


NetraScan.AI V4 has passed all four automated split-integrity checks. We can now preserve this version and prepare to retrain ViT-B/16.
All four checks passed: zero V4-family overlap, zero cross-split same-class pHash ≤4 pairs, zero detected Roboflow-lineage overlap, and zero missing image paths.

This is a substantial improvement over V1. It is still a similarity-group-aware split, not a patient-level split, and additional duplicate relationships could exist beyond our detection rules.

Step 26 — Save the V4 dataset manifests We'll preserve the original V1 CSVs and save V4 as a separate, reproducible dataset version.

In [ ]:

from pathlib import Path
import pandas as pd
import json

V4_DIR = ROOT / "NetraScan_Metadata" / "v4"
V4_DIR.mkdir(parents=True, exist_ok=True)

# Preserve the original metadata, plus V4 grouping and split assignments.
export_columns = [
    "source",
    "original_split",
    "filename",
    "class",
    "image_path",
    "sha256",
    "phash",
    "visual_family_id_v4",
    "visual_family_size_v4",
    "split_v4"
]

master_v4 = v4_df[export_columns].copy()

# Standardize column names for reuse in the training pipeline.
master_v4 = master_v4.rename(columns={
    "visual_family_id_v4": "visual_family_id",
    "visual_family_size_v4": "visual_family_size",
    "split_v4": "split"
})

master_v4.to_csv(
    V4_DIR / "netrascan_3class_master_v4.csv",
    index=False
)

for split_name in ["train", "val", "test"]:
    subset = master_v4[
        master_v4["split"] == split_name
    ].copy()

    subset.to_csv(
        V4_DIR / f"netrascan_{split_name}_v4.csv",
        index=False
    )

# Record the methodology for reproducibility.
split_config = {
    "version": "v4",
    "seed": 42,
    "target_split_ratio": {
        "train": 0.70,
        "val": 0.15,
        "test": 0.15
    },
    "grouping": [
        "Original visual-family relationships",
        "Roboflow filename augmentation lineage",
        "Global same-class pHash Hamming distance <= 4",
        "Union-Find connected components"
    ],
    "grouping_note": (
        "Similarity-based groups for split isolation; "
        "not verified patient identities."
    ),
    "total_images": len(master_v4),
    "visual_families": master_v4["visual_family_id"].nunique(),
    "split_counts": {
        k: int(v)
        for k, v in master_v4["split"].value_counts().items()
    }
}

with open(
    V4_DIR / "split_config_v4.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(split_config, f, indent=4)

print("V4 manifests saved successfully!\n")

for file in sorted(V4_DIR.iterdir()):
    print(file.name)

print("\nTotal images:", len(master_v4))
print("Visual families:", master_v4["visual_family_id"].nunique())
print("Split counts:", master_v4["split"].value_counts().to_dict())


Step 27 — Load V4 manifests into our notebook

In [ ]:

# STEP 27 — Load corrected V4 dataset

import pandas as pd

V4_DIR = ROOT / "NetraScan_Metadata" / "v4"

train_df = pd.read_csv(V4_DIR / "netrascan_train_v4.csv")
val_df = pd.read_csv(V4_DIR / "netrascan_val_v4.csv")
test_df = pd.read_csv(V4_DIR / "netrascan_test_v4.csv")

print("V4 TRAIN:", train_df.shape)
print("V4 VALIDATION:", val_df.shape)
print("V4 TEST:", test_df.shape)

print("\nClass distribution:")
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}")
    print(df["class"].value_counts())

print("\nMissing image paths:")
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing = (~df["image_path"].map(
        lambda p: Path(p).is_file()
    )).sum()
    print(f"{name}: {missing}")


In [ ]:
#Define the image transformations

from torchvision import transforms

# Training: mild augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=8),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

# Validation and testing: deterministic preprocessing
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

test_transform = val_transform

print("Training transforms: READY")
print("Validation transforms: READY")
print("Testing transforms: READY")


Step 28 — Recreate V4 datasets and DataLoaders 

In [ ]:

# STEP 28 — Build V4 datasets and DataLoaders

from torch.utils.data import DataLoader

BATCH_SIZE = 16
NUM_WORKERS = 0

# Reuse existing CataractDataset class and transforms
train_dataset = CataractDataset(
    train_df,
    transform=train_transform
)

val_dataset = CataractDataset(
    val_df,
    transform=val_transform
)

test_dataset = CataractDataset(
    test_df,
    transform=test_transform
)

# Recreate DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("V4 DATASET CONFIGURATION")
print("-" * 40)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

# Verify one training batch
images, labels = next(iter(train_loader))

print("\nBatch image shape:", images.shape)
print("Batch label shape:", labels.shape)

assert images.shape == (16, 3, 224, 224)
assert labels.shape == (16,)

print("\nPASS — V4 DataLoaders ready.")


V4 DataLoaders are fully operational. My corrected dataset is now ready for ViT-B/16 training on my RTX 3080 12 GB.

Step 29 — Initialize ViT-B/16 for V4 training
We'll initialize a fresh ImageNet-pretrained model, configure the optimizer and mixed precision, and create separate V4 output directories.

This ensures that my original V1 checkpoint remains untouched.

In [ ]:
#Cell 29 — Model and training configuration

import torch
import torch.nn as nn
import timm
from pathlib import Path

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Hardware
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Model configuration
MODEL_NAME = "vit_base_patch16_224"
NUM_CLASSES = 3

# Fresh ImageNet-pretrained ViT-B/16
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

# Training hyperparameters
EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 3

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Mixed precision for RTX GPU
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

# Separate V4 output directories
MODEL_DIR = ROOT / "models" / "vit_v4"
RESULTS_DIR = ROOT / "results" / "vit_v4"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "vit_v4_best.pth"

print("V4 TRAINING CONFIGURATION")
print("-" * 45)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("Model:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Classes:", NUM_CLASSES)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", BATCH_SIZE)
print("Mixed precision:", scaler.is_enabled())
print("Checkpoint path:", BEST_MODEL_PATH)

print("\nPASS — Fresh ViT-B/16 initialized for V4.")


Step 30 — Train ViT-B/16 on V4
We'll reuse the train_one_epoch() and validate_one_epoch() functions already defined in your notebook.

This cell will train for up to 10 epochs, save the best checkpoint, record metrics after every epoch, and stop early if validation macro-F1 fails to improve for three consecutive epochs.

In [ ]:

# STEP 30 — V4 ViT-B/16 training

import json
import time
import pandas as pd
import torch

# Verify training functions are available
assert callable(train_one_epoch)
assert callable(validate_one_epoch)

# Save experiment configuration
experiment_config = {
    "experiment": "vit_v4",
    "model": MODEL_NAME,
    "dataset_version": "v4",
    "num_classes": NUM_CLASSES,
    "class_names": ["normal", "immature", "mature"],
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "seed": SEED,
    "optimizer": "AdamW",
    "loss": "CrossEntropyLoss",
    "mixed_precision": scaler.is_enabled(),
    "selection_metric": "validation_macro_f1"
}

with open(
    RESULTS_DIR / "experiment_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(experiment_config, f, indent=4)

# Training state
history = []
best_val_f1 = -1.0
epochs_without_improvement = 0

print("Starting ViT-B/16 V4 training...")
print("Device:", DEVICE)
print("Checkpoint:", BEST_MODEL_PATH)

for epoch in range(1, EPOCHS + 1):

    start_time = time.time()

    # Training
    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE
    )

    # Validation
    val_loss, val_acc, val_f1 = validate_one_epoch(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    epoch_time = time.time() - start_time

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "train_macro_f1": float(train_f1),
        "val_loss": float(val_loss),
        "val_accuracy": float(val_acc),
        "val_macro_f1": float(val_f1),
        "epoch_time_seconds": round(epoch_time, 2)
    })

    # Save training history after every epoch
    pd.DataFrame(history).to_csv(
        RESULTS_DIR / "training_history.csv",
        index=False
    )

    print(
        f"\nEpoch {epoch}/{EPOCHS} | "
        f"Time: {epoch_time:.1f}s\n"
        f"Train: Loss={train_loss:.4f}, "
        f"Acc={train_acc:.4f}, F1={train_f1:.4f}\n"
        f"Val:   Loss={val_loss:.4f}, "
        f"Acc={val_acc:.4f}, F1={val_f1:.4f}"
    )

    # Select best checkpoint using validation macro-F1
    if val_f1 > best_val_f1 + 1e-6:

        best_val_f1 = float(val_f1)
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

        print("Best V4 checkpoint saved.")

    else:
        epochs_without_improvement += 1

        print(
            "No validation F1 improvement:",
            f"{epochs_without_improvement}/{PATIENCE}"
        )

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nV4 TRAINING COMPLETE")
print("-" * 45)
print("Epochs completed:", len(history))
print("Best validation macro-F1:", round(best_val_f1, 6))
print("Best checkpoint:", BEST_MODEL_PATH)
print("History:", RESULTS_DIR / "training_history.csv")


The most interesting observation is that the model achieved 100% validation accuracy after just one epoch, when training accuracy was only 89.24%.

This is possible because training accuracy includes predictions made while the model is still learning, and training uses augmentation. Nevertheless, such rapid convergence warrants further checks.

Why might this be happening?
There are several plausible explanations.

1. Residual dataset leakage: Our pHash ≤4 grouping catches many near-duplicates, but it cannot guarantee that all augmented versions of the same original image have been grouped.

2. Dataset shortcuts: The model may recognize differences in image backgrounds, acquisition conditions, preprocessing, or source-specific artifacts instead of cataract maturity.

3. An unusually easy validation set: The validation images may be extremely similar to the training distribution, even when they are not duplicates.

4. An evaluation implementation issue: We should verify that the validation function uses the correct labels, predictions, and DataLoader.

5. Genuine strong performance: The model may be learning useful cataract features, but we need independent evidence before concluding that.

The V4 checks rule out the specific relationships we tested. They do not rule out all these possibilities.

In [ ]:

# STEP 31 — Independently verify V4 validation predictions

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load the best V4 checkpoint, not the final epoch weights
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

all_true = []
all_pred = []
all_confidences = []

with torch.inference_mode():
    for images, labels in val_loader:

        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        predictions = probabilities.argmax(dim=1)
        confidences = probabilities.max(dim=1).values

        all_true.extend(labels.cpu().numpy())
        all_pred.extend(predictions.cpu().numpy())
        all_confidences.extend(confidences.cpu().numpy())

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_confidences = np.array(all_confidences)

print("V4 VALIDATION VERIFICATION")
print("-" * 45)

print("Total predictions:", len(all_pred))
print("Correct predictions:", int((all_true == all_pred).sum()))
print("Incorrect predictions:", int((all_true != all_pred).sum()))

print("\nAccuracy:", accuracy_score(all_true, all_pred))
print("Macro-F1:", f1_score(all_true, all_pred, average="macro"))

print("\nConfusion matrix:")
print(confusion_matrix(all_true, all_pred, labels=[0, 1, 2]))

print("\nClassification report:")
print(
    classification_report(
        all_true,
        all_pred,
        target_names=["normal", "immature", "mature"],
        labels=[0, 1, 2],
        zero_division=0
    )
)

print("Mean confidence:", round(float(all_confidences.mean()), 6))
print("Minimum confidence:", round(float(all_confidences.min()), 6))

assert len(all_pred) == len(val_dataset)
assert len(all_true) == len(val_df)

print("\nPASS — Independent validation inference completed.")


mean confidence was 99.41%, with a minimum confidence of 74.78%. This tells us the model is generally very confident, but confidence is not proof of correctness on unseen clinical data.

Step 32 — Check whether dataset source predicts the class Before moving to the test set, let's investigate a possible shortcut: whether the two source datasets have different class distributions.

If class labels are strongly associated with source, the model may partially exploit dataset artifacts rather than cataract features.

In [ ]:

# STEP 32 — Audit class distribution across dataset sources

import pandas as pd

# Load the full V4 manifest
audit_df = pd.read_csv(
    V4_DIR / "netrascan_3class_master_v4.csv"
)

print("SOURCE VS CLASS DISTRIBUTION")
print("-" * 55)

source_class_counts = pd.crosstab(
    audit_df["source"],
    audit_df["class"]
)

print("\nImage counts:")
print(source_class_counts)

print("\nClass percentages within each source:")
print(
    pd.crosstab(
        audit_df["source"],
        audit_df["class"],
        normalize="index"
    ).mul(100).round(2)
)

print("\nSOURCE VS SPLIT")
print("-" * 55)

print(
    pd.crosstab(
        audit_df["source"],
        audit_df["split"]
    )
)

print("\nSOURCE + CLASS VS SPLIT")
print("-" * 55)

print(
    pd.crosstab(
        [audit_df["source"], audit_df["class"]],
        audit_df["split"]
    )
)

print("\nPASS — Source distribution audit completed.")


Step 33 — Audit Dataset Katarak filenames and augmentation patterns

In [ ]:

# STEP 33 — Investigate Dataset Katarak augmentation filenames

import pandas as pd
import re

katarak_df = audit_df[
    audit_df["source"] == "dataset_katarak"
].copy()

# Remove the Roboflow export hash suffix
katarak_df["base_filename"] = (
    katarak_df["filename"]
    .str.replace(
        r"\.rf\.[0-9a-fA-F]+(?=\.[^.]+$)",
        "",
        regex=True
    )
    .str.lower()
)

# Extract common filename patterns
def get_filename_pattern(filename):
    pattern = re.sub(r"\d+", "<N>", filename)
    return pattern

katarak_df["filename_pattern"] = (
    katarak_df["base_filename"]
    .apply(get_filename_pattern)
)

print("DATASET KATARAK FILENAME AUDIT")
print("-" * 55)

print("Total images:", len(katarak_df))
print("Unique base filenames:", katarak_df["base_filename"].nunique())

print("\nMost common filename patterns:")
print(
    katarak_df["filename_pattern"]
    .value_counts()
    .head(15)
)

print("\nSample filenames:")
print(
    katarak_df[
        ["filename", "base_filename", "class", "split"]
    ].head(20).to_string(index=False)
)

# Check whether identical base filenames span splits
base_overlap = (
    katarak_df.groupby(
        ["class", "base_filename"]
    )["split"].nunique()
)

print(
    "\nBase filenames crossing splits:",
    int((base_overlap > 1).sum())
)

# Inspect largest V4 families from this source
print("\nLargest Katarak visual families:")
print(
    katarak_df.groupby("visual_family_id")
    .agg(
        images=("filename", "size"),
        classes=("class", "nunique"),
        splits=("split", "nunique")
    )
    .sort_values("images", ascending=False)
    .head(10)
)

print("\nPASS — Katarak filename audit completed.")


The key finding: 2,910 of 2,955 images use the aug_0_<number> naming pattern.

That strongly suggests the dataset contains augmented images, but the filenames don't tell us which images originated from the same eye photograph.

The 270 repeated base filenames are contained within individual splits, which is good. However, differently named augmented images could still share an original photograph.

Step 34 — Evaluate ViT-B/16 on the held-out V4 test set

In [ ]:

# STEP 34 — Evaluate V4 ViT-B/16 on the held-out test set

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load the best checkpoint selected using validation macro-F1
model.load_state_dict(
    torch.load(
        BEST_MODEL_PATH,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

test_true = []
test_pred = []
test_confidence = []

# Run inference on the held-out test dataset
with torch.inference_mode():
    for images, labels in test_loader:

        images = images.to(DEVICE)

        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)

        predictions = probabilities.argmax(dim=1)
        confidence = probabilities.max(dim=1).values

        test_true.extend(labels.cpu().numpy())
        test_pred.extend(predictions.cpu().numpy())
        test_confidence.extend(confidence.cpu().numpy())

test_true = np.array(test_true)
test_pred = np.array(test_pred)
test_confidence = np.array(test_confidence)

CLASS_NAMES = ["normal", "immature", "mature"]

# Calculate overall metrics
metrics = {
    "model": "ViT-B/16",
    "dataset": "V4",
    "test_images": len(test_true),
    "accuracy": accuracy_score(test_true, test_pred),
    "macro_precision": precision_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        test_true, test_pred, average="macro", zero_division=0
    ),
    "weighted_f1": f1_score(
        test_true, test_pred, average="weighted", zero_division=0
    )
}

print("V4 ViT-B/16 TEST EVALUATION")
print("-" * 50)

for key, value in metrics.items():
    print(f"{key}: {value}")

print("\nConfusion matrix:")
cm = confusion_matrix(test_true, test_pred, labels=[0, 1, 2])
print(cm)

print("\nClassification report:")
print(
    classification_report(
        test_true,
        test_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)

print("Mean confidence:", round(float(test_confidence.mean()), 6))
print("Minimum confidence:", round(float(test_confidence.min()), 6))

# Save metrics and predictions
pd.DataFrame([metrics]).to_csv(
    RESULTS_DIR / "test_metrics.csv",
    index=False
)

prediction_df = test_df.copy()
prediction_df["true_label"] = test_true
prediction_df["predicted_label"] = test_pred
prediction_df["confidence"] = test_confidence
prediction_df["correct"] = test_true == test_pred

prediction_df.to_csv(
    RESULTS_DIR / "test_predictions.csv",
    index=False
)

assert len(test_true) == 571

print("\nPASS — Test evaluation completed and results saved.")


What the model is getting wrong
The most important finding is that all 15 errors involve mature cataracts being classified as immature.

This is clinically relevant: underestimating cataract maturity could affect triage decisions. It also suggests that the model's weakest distinction is between immature and mature cataracts.

The test-set confidence average of 99.13% is high, but that does not establish calibration.

In [ ]:

# STEP 35 — Inspect ViT-B/16 V4 test misclassifications

import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import math

# Load the saved test predictions
predictions_df = pd.read_csv(
    RESULTS_DIR / "test_predictions.csv"
)

errors_df = predictions_df[
    predictions_df["correct"] == False
].copy()

print("Total misclassified images:", len(errors_df))

print("\nErrors by source:")
print(errors_df["source"].value_counts())

print("\nErrors by actual class:")
print(errors_df["class"].value_counts())

# Display all misclassified images
n = len(errors_df)
cols = 5
rows = math.ceil(n / cols)

if n > 0:
    fig, axes = plt.subplots(
        rows, cols,
        figsize=(16, rows * 3.5)
    )

    axes = axes.flatten()

    for ax, (_, row) in zip(axes, errors_df.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")

        ax.imshow(image)
        ax.set_title(
            f"Actual: {CLASS_NAMES[int(row['true_label'])]}\n"
            f"Pred: {CLASS_NAMES[int(row['predicted_label'])]}\n"
            f"Confidence: {row['confidence']:.3f}",
            fontsize=9
        )
        ax.axis("off")

    for ax in axes[n:]:
        ax.axis("off")

    plt.suptitle(
        "ViT-B/16 — V4 Test Misclassifications",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()
else:
    print("No misclassifications found.")

print("\nPASS — Misclassification inspection complete.")


The model made 15 mistakes, but many of those mistakes appear to be repeated or augmented versions of just a few underlying eye photographs.
Step 35 — What we discovered
- 13 errors originate from Cataract v01.
- 2 errors originate from Dataset Katarak.
- All 15 errors are mature cataracts predicted as immature.
- Several images look almost identical, especially the greenish-pupil photographs and the darker eye photographs.
This suggests that the 15 errors are not 15 independent failure cases. They're likely concentrated in a small number of visual families.

That matters because one difficult underlying image can produce several errors if multiple augmented versions are included in the test set.

Step 36 — Investigate the 15 errors by visual family
Before we generate our final figures, let's determine how many distinct V4 families account for these mistakes.

In [ ]:

# STEP 36 — Analyze ViT-B/16 errors by V4 visual family

import pandas as pd

errors_df = predictions_df[
    predictions_df["correct"] == False
].copy()

print("V4 MISCLASSIFICATION FAMILY ANALYSIS")
print("-" * 55)

print("Total misclassified images:", len(errors_df))

print(
    "Distinct misclassified visual families:",
    errors_df["visual_family_id"].nunique()
)

print("\nErrors grouped by visual family:")

family_errors = (
    errors_df.groupby(
        ["visual_family_id", "source", "class"]
    )
    .agg(
        error_count=("filename", "size"),
        mean_confidence=("confidence", "mean"),
        min_confidence=("confidence", "min")
    )
    .sort_values("error_count", ascending=False)
)

print(family_errors.to_string())

print("\nSource-specific test performance:")

source_performance = (
    predictions_df.groupby("source")
    .agg(
        total_images=("correct", "size"),
        correct_predictions=("correct", "sum")
    )
)

source_performance["errors"] = (
    source_performance["total_images"]
    - source_performance["correct_predictions"]
)

source_performance["accuracy"] = (
    source_performance["correct_predictions"]
    / source_performance["total_images"]
)

print(source_performance.to_string())

print("\nPASS — Error-family analysis completed.")


14 of the 15 mistakes belong to just two visual families. That's a much more specific finding than saying the model struggles generally with mature cataracts.

The high confidence for VFG_00090 is also noteworthy: the model repeatedly predicts immature with approximately 97% confidence on images labeled mature. This deserves investigation into both image ambiguity and label quality.

Step 37 — Generate and save ViT performance figures

In [ ]:

# STEP 37 — Generate and save ViT-B/16 V4 performance figures

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

history_df = pd.read_csv(
    RESULTS_DIR / "training_history.csv"
)

# Figure 1 — Training and validation accuracy
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    history_df["epoch"],
    history_df["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

ax.plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    marker="s",
    label="Validation accuracy"
)

ax.set_title("ViT-B/16 — V4 Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.set_xticks(history_df["epoch"])
ax.legend()
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "vit_v4_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Figure 2 — Held-out test confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Immature", "Mature"]
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title("ViT-B/16 — V4 Test Confusion Matrix")

fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "vit_v4_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("PASS — ViT-B/16 figures saved.")
print("Location:", RESULTS_DIR.resolve())
